# Extreme Condensation

Solution author: Cowile

goal: craft one 28x28 image + one 10-dim soft label such that a fixed ConvNet, trained on only that single example for 50 AdamW steps, classifies real MNIST digits as well as possible.

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split

DATA_ROOT = "/kaggle/input/extreme-condensation-aicc-round-4" # replace this with your dataset path
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 2026
BACKEND_SEED = 42
NUM_CLASSES = 10
IMG_SIZE = 28


def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print(f"Device: {device}")

Device: cuda


## Model

In [2]:
class ConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.InstanceNorm2d(32, affine=True),
            nn.ReLU(True),
            nn.AvgPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.InstanceNorm2d(32, affine=True),
            nn.ReLU(True),
            nn.AvgPool2d(2),
        )
        self.fc = nn.Linear(32 * 7 * 7, NUM_CLASSES)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        return self.fc(self.features(x).flatten(1))

## Data Loading

In [3]:
df = pd.read_csv(f"{DATA_ROOT}/train_labels.csv")
to_tensor = transforms.ToTensor()

all_images = [
    to_tensor(Image.open(os.path.join(DATA_ROOT, row.iloc[0])).convert("L"))
    for _, row in df.iterrows()
]

x_all = torch.stack(all_images)
y_all = torch.tensor(df.iloc[:, 1].values)

train_idx, val_idx = train_test_split(
    np.arange(len(y_all)),
    train_size=50000,
    random_state=SEED,
    stratify=y_all.numpy()
)

train_idx = torch.from_numpy(train_idx).long()
val_idx = torch.from_numpy(val_idx).long()

x_train = x_all[train_idx].contiguous()
y_train = y_all[train_idx].contiguous()
x_val = x_all[val_idx].contiguous()
y_val = y_all[val_idx].contiguous()

mean_image = x_train.mean(0).cpu()

print(f"Train: {len(y_train)} | Val: {len(y_val)}")

Train: 50000 | Val: 10000


## Helper Functions

A few utilities we'll need throughout:
- `inv_sigmoid`: maps pixel values in [0,1] to unconstrained logit space so we can optimize freely
- `soft_cross_entropy`: cross entropy against a soft label distribution instead of a hard class
- `train_and_eval`: reset weights, train for 50 AdamW steps on the synthetic example, then measure accuracy
- `score_on_train_subset`: evaluate on a random chunk of the training set to avoid overfitting to val

In [4]:
seed_everything(BACKEND_SEED)

ref_model = ConvNet().to(device)
ref_model.train()

# save the initial buffers and weights so we can reset the model for each eval
base_buffers = {name: buf.detach().clone() for name, buf in ref_model.named_buffers()}
base_state_dict = {k: v.cpu().clone() for k, v in ref_model.state_dict().items()}

def inv_sigmoid(x):
    x = x.clamp(1e-6, 1 - 1e-6)
    return x.log() - (1 - x).log()

def soft_cross_entropy(logits, targets):
    return -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

def train_and_eval(pixel_values, label_dist, x_test, y_test):
    net = ConvNet().to(device)
    net.load_state_dict(base_state_dict)
    net.train()

    optimizer = torch.optim.AdamW(net.parameters(), lr=0.01)

    x_syn = pixel_values.view(1, 1, IMG_SIZE, IMG_SIZE).to(device).clamp(0, 1)
    y_syn = label_dist.view(1, NUM_CLASSES).to(device).clamp(1e-8)
    y_syn = y_syn / y_syn.sum()

    for _ in range(50):
        optimizer.zero_grad(set_to_none=True)
        soft_cross_entropy(net(x_syn), y_syn).backward()
        optimizer.step()

    net.eval()
    num_correct = 0
    with torch.no_grad():
        for i in range(0, len(y_test), 2048):
            xb = x_test[i:i+2048].to(device)
            yb = y_test[i:i+2048].to(device)
            num_correct += (net(xb).argmax(dim=1) == yb).sum().item()

    return num_correct / len(y_test)

def score_on_train_subset(pixel_values, label_dist, n=8000):
    idx = torch.randperm(len(y_train))[:n]
    return train_and_eval(pixel_values, label_dist, x_train[idx], y_train[idx])

## Trajectory 

train a real model on real MNIST for a few steps and save parameter snapshots. Then cache the average gradient at each snapshot. These tell us what gradients should look like if learning is going well.

We'll use these cached gradients as targets in the gradient matching phase.

In [5]:
seed_everything(BACKEND_SEED)

expert_net = ConvNet().to(device)
expert_net.load_state_dict(base_state_dict)
expert_optimizer = torch.optim.AdamW(expert_net.parameters(), lr=0.01)

param_snapshots = [{name: p.detach().clone() for name, p in expert_net.named_parameters()}]

for step in range(1, 11):
    batch_idx = torch.randint(len(y_train), (512,))
    expert_optimizer.zero_grad(set_to_none=True)
    F.cross_entropy(
        expert_net(x_train[batch_idx].to(device)),
        y_train[batch_idx].to(device)
    ).backward()
    expert_optimizer.step()
    param_snapshots.append({name: p.detach().clone() for name, p in expert_net.named_parameters()})

# for each snapshot, compute the average gradient over 20 random batches, this gives us a stable estimate of what the true gradient looks like at each checkpoint
cached_gradients = []

for snap_idx in range(len(param_snapshots)):
    snapshot = param_snapshots[snap_idx]
    avg_grads = {name: torch.zeros_like(p) for name, p in snapshot.items()}

    for _ in range(20):
        batch_idx = torch.randint(len(y_train), (512,))
        xb = x_train[batch_idx].to(device)
        yb = y_train[batch_idx].to(device)

        params = {name: v.clone().requires_grad_(True) for name, v in snapshot.items()}
        loss = F.cross_entropy(
            functional_call(ref_model, {**base_buffers, **params}, (xb,)),
            yb
        )
        grads = torch.autograd.grad(loss, list(params.values()))

        for (name, _), g in zip(params.items(), grads):
            avg_grads[name] += g.detach() / 20

    cached_gradients.append(avg_grads)

print(f"{len(param_snapshots)} expert snapshots ready")

11 expert snapshots ready


## Gradient Matching Loss + Random Search

**Gradient matching**: for a given snapshot, compute the gradient your synthetic image produces and compare it to the cached real gradient using cosine similarity. The FC layer gets 10x weight because it controls class predictions most directly.

**Random search**: gradient matching gives a decent starting point but optimizes a proxy. To directly improve accuracy, we do zeroth-order random search:
- Sample a random perturbation direction
- Try it in both +/- directions 
- Keep if train subset accuracy improves

In [6]:
def gradient_matching_loss(snap_idx, x_syn, y_syn):
    params = {name: v.clone().requires_grad_(True) for name, v in param_snapshots[snap_idx].items()}
    loss = soft_cross_entropy(
        functional_call(ref_model, {**base_buffers, **params}, (x_syn,)),
        y_syn
    )
    syn_grads = torch.autograd.grad(loss, list(params.values()), create_graph=True)

    total_loss = torch.tensor(0.0, device=device)
    for syn_g, (name, _) in zip(syn_grads, params.items()):
        real_g = cached_gradients[snap_idx][name]
        cos_sim = F.cosine_similarity(
            syn_g.flatten().unsqueeze(0),
            real_g.flatten().unsqueeze(0)
        )

        layer_weight = 10.0 if "fc" in name else 1.0
        total_loss = total_loss + layer_weight * (1.0 - cos_sim)

    return total_loss


def random_search(init_pix_logits, init_lab_logits, num_steps=2500):
    best_pixels = torch.sigmoid(init_pix_logits).flatten()
    best_labels = F.softmax(init_lab_logits, dim=0)
    best_score = score_on_train_subset(best_pixels, best_labels)

    pix_logits = init_pix_logits.clone()
    lab_logits = init_lab_logits.clone()

    PIX_SIGMA_MAX = 0.3
    PIX_SIGMA_MIN = 3e-3
    LAB_SIGMA_MAX = 0.15
    LAB_SIGMA_MIN = 1.5e-3
    RESTART_PERIOD = 250

    stale_steps = 0

    for step in range(1, num_steps + 1):
        t = ((step - 1) % RESTART_PERIOD) / RESTART_PERIOD
        cosine_weight = 0.5 * (1 + np.cos(np.pi * t))

        pix_sigma = PIX_SIGMA_MIN + (PIX_SIGMA_MAX - PIX_SIGMA_MIN) * cosine_weight
        lab_sigma = LAB_SIGMA_MIN + (LAB_SIGMA_MAX - LAB_SIGMA_MIN) * cosine_weight

        pix_noise = pix_sigma * torch.randn(IMG_SIZE * IMG_SIZE)
        lab_noise = lab_sigma * torch.randn(NUM_CLASSES)

        improved = False
        for sign in [1, -1]:
            candidate_pixels = torch.sigmoid(pix_logits + sign * pix_noise)
            candidate_labels = F.softmax(lab_logits + sign * lab_noise, dim=0)
            score = score_on_train_subset(candidate_pixels, candidate_labels)

            if score > best_score:
                pix_logits = pix_logits + sign * pix_noise
                lab_logits = lab_logits + sign * lab_noise
                best_pixels = torch.sigmoid(pix_logits).flatten()
                best_labels = F.softmax(lab_logits, dim=0)
                best_score = score
                improved = True
                stale_steps = 0
                break

        if not improved:
            stale_steps += 1

        if step % 500 == 0:
            val_score = train_and_eval(best_pixels, best_labels, x_val, y_val)
            print(f"    RS {step}: train={best_score:.4f} val={val_score:.4f} stale={stale_steps}")

        if stale_steps >= 600:
            print(f"    early stop at {step}")
            break

    return best_pixels, best_labels, best_score

## Optimization Loop

12 restarts each:
1. Initializes pixel logits near the mean MNIST image (small noise added)
2. Run 200 steps of gradient matching to get a reasonable starting point
3. Switch to random search on training subsets

Best restart by val accuracy wins.

In [7]:
rng = np.random.default_rng(SEED)
candidates = []
num_snapshots = len(param_snapshots)

for restart_id in range(12):
    print(f"\n=== R{restart_id} ===")
    seed_everything(SEED + restart_id * 1000)

    # initialize near the mean MNIST image with a bit of noise
    init_pix = (mean_image + 0.05 * torch.randn_like(mean_image)).clamp(0, 1)
    pix_logits = nn.Parameter(inv_sigmoid(init_pix).to(device))
    lab_logits = nn.Parameter(0.02 * torch.randn(NUM_CLASSES, device=device))

    gm_optimizer = torch.optim.Adam([
        {"params": [pix_logits], "lr": 0.01},
        {"params": [lab_logits], "lr": 0.008}
    ])

    for step in range(1, 201):
        gm_optimizer.zero_grad(set_to_none=True)

        x_syn = torch.sigmoid(pix_logits).view(1, 1, IMG_SIZE, IMG_SIZE)
        y_syn = F.softmax(lab_logits, dim=0).view(1, NUM_CLASSES)

        gm_loss = sum(
            gradient_matching_loss(rng.integers(num_snapshots), x_syn, y_syn)
            for _ in range(3)
        ) / 3

        # small TV regularization to keep the image smooth
        tv_loss = (
            (x_syn[:, :, 1:] - x_syn[:, :, :-1]).pow(2).mean()
            + (x_syn[:, :, :, 1:] - x_syn[:, :, :, :-1]).pow(2).mean()
        )

        (gm_loss + 1e-4 * tv_loss).backward()
        nn.utils.clip_grad_norm_([pix_logits, lab_logits], 1.0)
        gm_optimizer.step()

        with torch.no_grad():
            pix_logits.clamp_(-8, 8)
            lab_logits.clamp_(-15, 15)

    gm_pixels = torch.sigmoid(pix_logits).detach().flatten().cpu()
    gm_labels = F.softmax(lab_logits.detach(), dim=0).cpu()
    gm_acc = train_and_eval(gm_pixels, gm_labels, x_val, y_val)
    print(f"  GM: {gm_acc:.4f}")

    # random search
    rs_pixels, rs_labels, rs_train_score = random_search(
        pix_logits.detach().cpu().flatten(),
        lab_logits.detach().cpu()
    )
    rs_val_score = train_and_eval(rs_pixels, rs_labels, x_val, y_val)
    candidates.append((rs_pixels, rs_labels, rs_val_score, restart_id))
    print(f"  R{restart_id}: train={rs_train_score:.4f} val={rs_val_score:.4f}")


=== R0 ===
  GM: 0.0939
    RS 500: train=0.2169 val=0.2178 stale=263
    early stop at 837
  R0: train=0.2169 val=0.2178

=== R1 ===
  GM: 0.1003
    RS 500: train=0.2050 val=0.1943 stale=3
    RS 1000: train=0.2050 val=0.1943 stale=503
    early stop at 1097
  R1: train=0.2050 val=0.1943

=== R2 ===
  GM: 0.0947
    RS 500: train=0.1681 val=0.1593 stale=257
    RS 1000: train=0.1691 val=0.1650 stale=254
    RS 1500: train=0.1711 val=0.1714 stale=255
    early stop at 1845
  R2: train=0.1711 val=0.1714

=== R3 ===
  GM: 0.1198
    RS 500: train=0.1916 val=0.1820 stale=9
    RS 1000: train=0.2077 val=0.1996 stale=416
    early stop at 1184
  R3: train=0.2077 val=0.1996

=== R4 ===
  GM: 0.1226
    RS 500: train=0.2165 val=0.2105 stale=381
    early stop at 719
  R4: train=0.2165 val=0.2105

=== R5 ===
  GM: 0.1104
    RS 500: train=0.1861 val=0.1793 stale=164
    early stop at 936
  R5: train=0.1861 val=0.1793

=== R6 ===
  GM: 0.0696
    RS 500: train=0.1993 val=0.2081 stale=4
    RS

## Submission

Pick the best restart by val score, normalize the label, and write to `submission.csv`.

In [8]:
candidates.sort(key=lambda x: x[2], reverse=True)

print("--- Ranking ---")
for rank, (_, _, score, restart_id) in enumerate(candidates):
    print(f"  {rank+1}. R{restart_id} {score:.4f}")

best_pixels, best_labels = candidates[0][0], candidates[0][1]

pix_np = best_pixels.numpy().astype(np.float32).clip(0, 1)
lab_np = best_labels.numpy().astype(np.float32).clip(0, None)
lab_np = lab_np / lab_np.sum()

submission_row = {"id": 0}
submission_row.update({f"p{i}": float(pix_np[i]) for i in range(IMG_SIZE * IMG_SIZE)})
submission_row.update({f"l{i}": float(lab_np[i]) for i in range(NUM_CLASSES)})

pd.DataFrame([submission_row]).to_csv("submission.csv", index=False)
print(f"Saved | pix [{pix_np.min():.4f}, {pix_np.max():.4f}] | labels: {lab_np.round(3)}")

--- Ranking ---
  1. R11 0.2910
  2. R8 0.2333
  3. R10 0.2181
  4. R9 0.2180
  5. R0 0.2178
  6. R4 0.2105
  7. R6 0.2081
  8. R3 0.1996
  9. R7 0.1988
  10. R1 0.1943
  11. R5 0.1793
  12. R2 0.1714
Saved | pix [0.0000, 0.9503] | labels: [0.063 0.087 0.055 0.176 0.087 0.181 0.14  0.034 0.097 0.08 ]


## References

Papers that directly influenced this approach:

- Zhao et al. (2021) [Dataset Condensation with Gradient Matching](https://arxiv.org/abs/2006.05929) - the Gradient Matching technique used for initialization. We use cosine similarity instead of L2.

- Cazenavette et al. (2022) [Dataset Distillation by Matching Training Trajectories](https://arxiv.org/abs/2203.11932) - we save parameter snapshots from a real training run and match gradients at those checkpoints rather than at random init.

- Wang et al. (2018) [Dataset Distillation](https://arxiv.org/abs/1811.10959) - the original paper that framed this as a bilevel optimization problem. We ended up not using this approach directly, but still relevant.